# Titanic Exercise

**Notebook 6 of 6, and optional.** Everything from notebooks 01 to 05, on a dataset you have
not seen:
the passenger list of the Titanic, which sank on its first voyage in April 1912. **Predict
whether a passenger survived.**

This is the full list of 1,309 passengers. Better-known versions of it leave out a few
columns, and this one keeps them in. **At least one of those columns leaks the answer.**
Part of the job is to find it before it fools you.

The steps below walk the workflow in order. Each has a cell marked `# your code` for you to
fill in; the steps build on each other, so work top to bottom.

**The columns:**

| Column | Meaning |
| :-- | :-- |
| `pclass` | ticket class: 1 = first, 2 = second, 3 = third |
| `survived` | **the target**: 1 = survived, 0 = died |
| `name` | name, including a title such as Mr, Mrs, Miss or Master |
| `sex` | `male` or `female` |
| `age` | age in years; fractions for infants |
| `sibsp` | number of siblings and spouses aboard |
| `parch` | number of parents and children aboard |
| `ticket` | ticket number |
| `fare` | ticket price, in British pounds |
| `cabin` | cabin number |
| `embarked` | port of boarding: C = Cherbourg, Q = Queenstown, S = Southampton |
| `boat` | lifeboat number or letter |
| `body` | identification number given to a recovered body |
| `home.dest` | home town and destination |

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
titanic = pd.read_csv("../data/titanic.csv")
titanic.head()

## Step 1: Features, target, split

Separate the table into features `X` and target `y`, and check that the target is not
among the features. Then split with `test_size=0.25`, `random_state=42` and stratification.
**Everything from here on looks at the training set only**, until the final score.

In [ ]:
y = titanic["survived"]
X = titanic.drop(columns="survived")
print("target among features:", "survived" in X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("train:", X_train.shape, " test:", X_test.shape)

## Step 2: A first look, at the training set

How many training passengers survived, as a share? And how many values are missing in
each column?

In [ ]:
print(f"survival rate: {y_train.mean():.3f}")
X_train.isna().sum()

<details><summary>
Click here to compare your answer...
</summary>

About **38 percent** survived. `cabin`, `boat`, `body` and `home.dest` are mostly empty,
and about a fifth of the ages are missing. One fare is missing too. `embarked` has no gaps
in this training set, but two passengers in the test set have no recorded port.
</details>

## Step 3: Two baselines

Compute the test accuracy of two rules that need no model:

1. **Everyone died**, the majority class of the training set.
2. **Women survived and men died.** "Women and children first" was the rule for the
   lifeboats, so this is a baseline from domain knowledge.

These are the numbers your model has to beat. Scoring a fixed rule on the test set is
fine: nothing was chosen by looking at it.

In [ ]:
majority = y_train.mode()[0]
print(f"everyone died: {(y_test == majority).mean():.3f}")

women_survive = (X_test["sex"] == "female").astype(int)
print(f"women survived: {accuracy_score(y_test, women_survive):.3f}")

<details><summary>
Click here to compare your answer...
</summary>

About **62 percent** for "everyone died" and about **80 percent** for "women survived".
The second one is a strong baseline: sex alone explains most of who survived. Keep that
80 in mind.
</details>

## Step 4: Find the leak

Two columns are suspicious: `boat` and `body`. For each, compare the survival rate of
training passengers where the column **is filled in** with those where it is empty.

<details><summary>
Click here for a hint...
</summary>

`X_train["boat"].notna()` is `True` where a boat is recorded. `pd.crosstab` of that against
`y_train`, with `normalize="index"`, gives the survival rate in each group.
</details>

In [ ]:
print(pd.crosstab(X_train["boat"].notna(), y_train, normalize="index").round(3))
print()
print(pd.crosstab(X_train["body"].notna(), y_train, normalize="index").round(3))

> **Exercise:**
>
> No code needed. Answer the question from notebooks 01 and 05 for both columns: **was the
> value known at the moment a prediction would have to be made**, before the sinking? What
> do you do with the two columns?

Your answer here:

<details><summary>
Click here to compare your answer...
</summary>

A passenger with a recorded **boat** survived about 98 percent of the time; one with a
recorded **body** number never did. Both are **target leakage**: a boat number is written
down for people who got into a lifeboat, and a body number for people whose body was
recovered after the sinking. Neither exists before the outcome. **Drop both.**

The other text columns are not leaks. `name`, `ticket`, `cabin` and `home.dest` are left
out here only to keep the model simple.
</details>

## Step 5: Prepare the features

The function below turns the raw columns into numbers a model can use: it fills missing
ages and fares, encodes `sex` as `male` (1 or 0), and one-hot encodes `embarked`, as in
notebook 04. `boat` and `body` are not used.

Its two fill values are **arguments**, on purpose: they are computed from data, so they
must come from the training set, and the same values must then be used for the test set.

In [ ]:
def prepare(df, age_fill, fare_fill):
    """Turn raw passenger rows into numeric features.

    Parameters
    ----------
    df : pandas.DataFrame
        Passenger rows with the original columns.
    age_fill : float
        Value for a missing age, computed on the training set.
    fare_fill : float
        Value for a missing fare, computed on the training set.

    Returns
    -------
    pandas.DataFrame
        One row per passenger: class, age, family counts, fare, sex as `male`, and one
        column per port of embarkation.
    """
    features = df[["pclass", "age", "sibsp", "parch", "fare"]].copy()
    features["age"] = features["age"].fillna(age_fill)
    features["fare"] = features["fare"].fillna(fare_fill)
    features["male"] = (df["sex"] == "male").astype(int)

    # one column per port; a passenger with no recorded port gets 0 in all three
    ports = pd.get_dummies(df["embarked"], prefix="embarked", dtype=int)
    return features.join(ports)

Compute the two fill values, the **median** age and the **median** fare, on the right set,
and use `prepare` to build `X_train_ready` and `X_test_ready`. Check that both have the
same columns.

In [ ]:
age_fill = X_train["age"].median()
fare_fill = X_train["fare"].median()
print("fill values:", age_fill, round(fare_fill, 2))

X_train_ready = prepare(X_train, age_fill, fare_fill)
X_test_ready = prepare(X_test, age_fill, fare_fill)

print("same columns:", list(X_train_ready.columns) == list(X_test_ready.columns))
X_train_ready.head()

## Step 6: Train and evaluate

Scale the features, with the scaler fitted on the training set, and train a logistic
regression. Print its training and test accuracy, and compare them with both baselines.

In [ ]:
scaler = StandardScaler().fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)

model = LogisticRegression().fit(X_train_scaled, y_train)

print(f"train accuracy: {model.score(X_train_scaled, y_train):.3f}")
print(f"test accuracy:  {model.score(X_test_scaled, y_test):.3f}")
pd.Series(model.coef_[0], index=X_train_ready.columns).round(2)

<details><summary>
Click here to compare your answer...
</summary>

About **79 percent** on the training set and **80 percent** on the test set: far above
"everyone died", and **level with "women survived"**, 0.3 points ahead, which on 328
test passengers is a single person.

That is an honest result, not a failure. The coefficients show why: `male` is the largest,
and negative, so the model has largely rediscovered the baseline. Class and age
come next, both negative: first class and youth helped. A strong baseline is hard to beat,
and knowing that is worth more than a flattering number.
</details>

## Step 7: What the leak would have done

Add one column to both prepared sets: `has_boat`, 1 where a boat is recorded and 0
otherwise. Scale, train and score exactly as in step 6. What happens to the test accuracy,
and to the coefficients?

In [ ]:
X_train_leak = X_train_ready.assign(has_boat=X_train["boat"].notna().astype(int))
X_test_leak = X_test_ready.assign(has_boat=X_test["boat"].notna().astype(int))

scaler_leak = StandardScaler().fit(X_train_leak)
model_leak = LogisticRegression().fit(scaler_leak.transform(X_train_leak), y_train)

print(f"test accuracy with the leak: {model_leak.score(scaler_leak.transform(X_test_leak), y_test):.3f}")
pd.Series(model_leak.coef_[0], index=X_train_leak.columns).round(2)

<details><summary>
Click here to compare your answer...
</summary>

About **97 percent**, and `has_boat` dwarfs every other coefficient: all the checklist's
warning signs from notebook 05 at once. It would be the best-looking model in this
repository, and it could not predict anything, because for a passenger about to board, the
column is always empty.
</details>

## Stretch (optional): one more feature

"Women and children first". The model knows the sex, but a boy is a `male` like any man.
The `name` column holds the title **Master**, which was used for boys.

Add a column `master`, 1 where the name contains `"Master."`, to both prepared sets, and
train as in step 6. Decide whether it helps by the **training** accuracy first, then check
the test accuracy once.

<details><summary>
Click here for a hint...
</summary>

`X_train["name"].str.contains("Master.", regex=False)` gives `True` or `False` per
passenger; `.astype(int)` turns it into 1 or 0.
</details>

In [ ]:
X_train_master = X_train_ready.assign(
    master=X_train["name"].str.contains("Master.", regex=False).astype(int)
)
X_test_master = X_test_ready.assign(
    master=X_test["name"].str.contains("Master.", regex=False).astype(int)
)

scaler_master = StandardScaler().fit(X_train_master)
model_master = LogisticRegression().fit(scaler_master.transform(X_train_master), y_train)

print(f"train accuracy: {model_master.score(scaler_master.transform(X_train_master), y_train):.3f}")
print(f"test accuracy:  {model_master.score(scaler_master.transform(X_test_master), y_test):.3f}")

<details><summary>
Click here to compare your answer...
</summary>

Training accuracy rises from about 79 to 81 percent, and the test accuracy, checked once,
to about **82 percent**: now clearly ahead of the "women survived" baseline. The feature
came from understanding the problem, not from the test set, which is how a model should
improve.
</details>

## Summary

| Model | Test accuracy |
| :-- | --: |
| Everyone died | 62% |
| Women survived | 80% |
| Logistic regression | 80% |
| Logistic regression with the leaking `has_boat` | 97%, and useless |
| Logistic regression with `master` (stretch) | 82% |

The whole workflow of this repository, in the order you just used it:

1. Separate **features and target**, and split before anything else.
2. Explore **the training set only**.
3. Set **baselines**, including one from domain knowledge.
4. Check every feature for **leakage**: known at prediction time, not computed from the
   target.
5. Compute fill values and scalers **on the training set**, apply them to both.
6. Train, and score on the test set **once**.